# 12 — Perbaikan tekstur melon dengan discriminator interior dan SKConv

Notebook Kaggle/local untuk eksperimen **baru**, tidak mengubah run G0X-EMA lama.

Prioritas 1: discriminator tambahan membaca patch **32 × 32 piksel asli** dari bagian dalam mask buah (≥95% mask), tanpa resize dan tanpa latar hitam buatan. Domain A=unripe, B=ripe; fake_A memakai mask sumber B. Grayscale adalah hipotesis untuk menekankan tekstur, sementara discriminator utama tetap RGB. Tidak ada target tekstur pixel-wise antara ripe dan unripe yang tidak berpasangan.

Prioritas 2: SK-inspired block dengan cabang 3×3 dan dilated 3×3, fusi softmax per channel, ditempatkan setelah downsampling generator. Ini adaptasi ringan; **bukan reproduksi persis Tea CycleGAN/SCU-CGAN dan belum terbukti menaikkan AP**.

Default: pilot 10+10 epoch, baseline vs ROI, satu seed. Untuk ablation lengkap gunakan baseline / ROI / SK / ROI+SK pada jadwal identik. Full: 60+60 epoch, seluruh model dilatih dari awal. Pilot tidak dibandingkan dengan baseline lama 120 epoch. Satu seed generator belum mengukur stabilitas lintas seed.

## Input Kaggle
1. Upload dan attach `texture_roi_sk_code.zip` sebagai dataset kode (boleh tetap zip atau diekstrak Kaggle).
2. Attach dataset beku `melon-dataset-regrouped-v4` dan assets yang memuat `reviewed_masks.json`, sama seperti notebook 08.
3. Pilih GPU. Run All menjalankan **pilot** secara berurutan pada satu GPU; tidak mengirim job Kaggle dari komputer lokal.
4. Kode paket sudah mencakup `cyclegan` dan `14_Benchmark_Kaggle`. Tidak memerlukan instalasi model baru/download bobot pretrained.

Data test tidak dibaca. Output panel merupakan diagnostik **train**, bukan evaluasi generalisasi. Gambar yang tampak lebih tajam bisa mengandung noise; penilaian tekstur harus dibandingkan dengan unripe nyata.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, zipfile, hashlib

# Untuk lokal, isi ketiga path. Kosong = pencarian mount Kaggle.
CODE_SOURCE = None     # folder code yang berisi cyclegan/ dan 14_Benchmark_Kaggle/
DATASET_PATH = None    # folder manifest.csv dataset beku regrouped v4
ASSETS_PATH = None     # folder reviewed_masks.json
EXISTING_POOL = None   # opsional: curated 298-crop pool lengkap, termasuk PNG dan mask
WORK = Path('/kaggle/working/texture_experiment') if Path('/kaggle').exists() else Path.cwd() / 'texture_experiment'
STAGE = 'pilot'        # 'pilot' = 10+10; 'full' = 60+60, rerun from scratch
ARMS = ['baseline', 'roi']
# Prioritas 2: ARMS = ['baseline', 'roi', 'sk', 'roi_sk']
SEEDS = [11]          # replikasi final dapat memakai [11, 23, 37]
LAMBDA_TEXTURE = 0.5  # titik awal eksperimen, bukan angka optimal dari paper
TEXTURE_SIZE = 32  # CPU preflight: 64/48 fail on 2 masks; 32 passes all 298
TEXTURE_GRAY = 1      # 1=grayscale, 0=RGB; perubahan ini harus menjadi run terpisah
BUDGET_MINUTES_PER_RUN = 30
PANEL_EPOCHS = ['latest']   # contoh full: ['40', '80', '120']; tambahkan sebelum run
PANEL_WEIGHTS = ['raw', 'ema']
assert STAGE in ('pilot','full')
assert set(ARMS) <= {'baseline','roi','sk','roi_sk'} and len(set(ARMS)) == len(ARMS)
assert ARMS and SEEDS and len(set(SEEDS)) == len(SEEDS)
WORK.mkdir(parents=True, exist_ok=True)
INPUT = Path('/kaggle/input')

def unique_parent(marker):
    hits = sorted({p.parent for p in INPUT.rglob(marker)}) if INPUT.exists() else []
    if len(hits) != 1:
        raise ValueError(f'Isi path manual: marker {marker}, kandidat: {hits}')
    return hits[0]

if CODE_SOURCE is None:
    configs = list(INPUT.rglob('texture_roi.json')) if INPUT.exists() else []
    if len(configs) == 1:
        CODE_SOURCE = configs[0].parents[2]
    elif not configs:
        archives = list(INPUT.rglob('texture_roi_sk_code.zip')) if INPUT.exists() else []
        if len(archives) != 1:
            raise ValueError('Attach paket kode terbaru, atau isi CODE_SOURCE')
        unpack = WORK / 'unpacked_code'
        with zipfile.ZipFile(archives[0]) as z:
            for item in z.infolist():
                target = (unpack/item.filename).resolve()
                if not target.is_relative_to(unpack.resolve()):
                    raise ValueError('Path zip tidak aman')
            z.extractall(unpack)
        CODE_SOURCE = unpack/'code'
    else:
        raise ValueError('Lebih dari satu versi kode terpasang; isi CODE_SOURCE')
DATASET = Path(DATASET_PATH) if DATASET_PATH else unique_parent('manifest.csv')
ASSETS = Path(ASSETS_PATH) if ASSETS_PATH else unique_parent('reviewed_masks.json')
CODE = WORK/'code'
# Tidak menimpa snapshot kode run yang sudah ada.
if CODE.exists():
    raise FileExistsError(f'{CODE} sudah ada; pakai WORK baru untuk eksperimen baru')
shutil.copytree(CODE_SOURCE, CODE, ignore=shutil.ignore_patterns('__pycache__','*.pyc','.git'))
HERE = CODE/'cyclegan'
assert (HERE/'config/texture_roi.json').exists()
assert (CODE/'14_Benchmark_Kaggle/src/common.py').exists()
os.environ.update(MELON_DATASET=str(DATASET.resolve()), MELON_ASSETS=str(ASSETS.resolve()))
sys.path.insert(0,str(HERE))
ENV = {**os.environ, 'PYTHONPATH': str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2'}
def run(*args, env=None):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=env or ENV,check=True)
import torch
print('torch',torch.__version__,'CUDA',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Training membutuhkan GPU. Uji unit CPU dapat dijalankan terpisah.')
print(torch.cuda.get_device_name(0))


## Uji implementasi dan verifikasi data
Uji CPU memeriksa mask/gradient, training dua cabang discriminator, SKConv batch satu, EMA serta checkpoint inference. Uji AMP di bawah berjalan jika GPU tersedia. Model kecil pada uji ini bukan eksperimen ilmiah.


In [ ]:
run(HERE/'tests/test_texture.py')
CROPS = Path(EXISTING_POOL).resolve() if EXISTING_POOL else WORK/'crops256_g0x_v2'
if EXISTING_POOL is None:
    SOURCE_CROPS = WORK/'crops256'
    run(HERE/'tools/build_crops.py','--out',SOURCE_CROPS,'--dataset',DATASET)
    run(HERE/'tools/prepare_g0x_data.py','--source',SOURCE_CROPS,'--out',CROPS)
from melon_cyclegan.training_data import verify
from melon_cyclegan.dataset_impl import MelonRoiData, load_pair
from melon_cyclegan.texture import patch_indices
manifest = verify(CROPS,'g0x_train_v2')
assert manifest['crops_digest'] == '8cec926af44bd080b0b3e052a7b92afa2eedf2e051c361e2a7e3bf9045b9afe5'
data = MelonRoiData(CROPS,no_flip=True)
failures=[]
for domain,paths in [('A',data.A_paths),('B',data.B_paths)]:
    for path in paths:
        _,mask = load_pair(path,data.mask_for(path,domain))
        try:
            patch_indices(mask[None],TEXTURE_SIZE,16,4,.95,random=False)
        except ValueError as e:
            failures.append({'id':path.stem,'domain':domain,'error':str(e)})
(WORK/'patch_preflight.json').write_text(json.dumps({'images':len(data.A_paths)+len(data.B_paths),
    'failures':failures,'patch_size':TEXTURE_SIZE,'coverage':.95},indent=2))
assert not failures, f'{len(failures)} mask tidak memiliki patch interior. Periksa patch_preflight.json; jangan turunkan threshold diam-diam.'
print('Pool:',len(data.A_paths),'unripe;',len(data.B_paths),'ripe. Semua mask lolos.')


## Bekukan konfigurasi dan jalankan training
Semua arm memakai data, seed, jadwal dan augmentasi global yang sama. Tambahan discriminator memakai patch baru tanpa replay buffer agar mask dan citra tidak tertukar. Local GAN loss memiliki bobot 0,5 pada generator dan discriminator. Ukuran 32, coverage 0,95, empat patch, grayscale dan bobot tersebut adalah keputusan eksperimen yang dicatat, bukan hasil optimasi.

Training serial mengurangi risiko kehabisan VRAM. Waktu tambahan belum dibenchmark. Batas waktu yang terlampaui membatalkan kelengkapan eksperimen; hasil parsial tidak dianggap selesai. Log ditampilkan saat training dan disimpan.


In [ ]:
CFGDIR = WORK/'configs'
CFGDIR.mkdir(exist_ok=False)
CKPT = WORK/'checkpoints'
JOBS=[]
for arm in ARMS:
    for seed in SEEDS:
        settings=json.loads((HERE/f'config/texture_{arm}.json').read_text())
        name=f'texture_{STAGE}_{arm}_seed{seed}'
        settings.update(experiment=name,seed=seed)
        if STAGE == 'pilot':
            settings.update(n_epochs=10,n_epochs_decay=10)
        if 'roi' in arm:
            settings.update(lambda_texture=LAMBDA_TEXTURE,texture_gray=TEXTURE_GRAY,texture_size=TEXTURE_SIZE)
        path=CFGDIR/f'{name}.json'
        path.write_text(json.dumps(settings,indent=2)+'\n')
        run(HERE/'tools/train_cyclegan.py','--config',path,'--dataroot',CROPS,
            '--checkpoints-dir',CKPT,'--dry-run')
        JOBS.append({'arm':arm,'seed':seed,'name':name,'config':str(path)})
code_hashes={str(p.relative_to(CODE)):hashlib.sha256(p.read_bytes()).hexdigest()
             for p in CODE.rglob('*.py')}
(WORK/'experiment_plan.json').write_text(json.dumps(dict(stage=STAGE,jobs=JOBS,
    code_sha256=code_hashes,torch=torch.__version__,cuda=torch.version.cuda,
    crops_digest=manifest['crops_digest'],test_used=False,
    metric_selection='future real validation AP unripe + mAP50:95; train panels are diagnostic'),indent=2))

for job in JOBS:
    name=job['name']
    command=[sys.executable,str(HERE/'tools/train_cyclegan.py'),'--config',job['config'],
             '--dataroot',str(CROPS),'--checkpoints-dir',str(CKPT),'--',
             '--time_budget_minutes',str(BUDGET_MINUTES_PER_RUN)]
    started=time.time()
    with (WORK/f'{name}.log').open('w') as log:
        proc=subprocess.Popen(command,cwd=HERE,env={**ENV,'CUDA_VISIBLE_DEVICES':'0','PYTHONUNBUFFERED':'1'},
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:
                log.write(line); log.flush(); print(line,end='')
            rc=proc.wait()
        except BaseException:
            proc.terminate(); proc.wait(); raise
    if rc: raise RuntimeError(f'{name} gagal; lihat log')
    summary=json.loads((CKPT/name/'loss.json').read_text())
    if summary['stopped']: raise RuntimeError(f'{name}: jadwal terpotong {summary["stopped"]}')
    print(name,'selesai, menit:',round((time.time()-started)/60,1))


## Panel tekstur pada input yang sama
Setiap halaman memperlihatkan: sumber ripe, hasil dengan soft mask training, hasil dengan hard mask ekspor, patch sumber, patch sintetis, dan referensi unripe nyata **tidak berpasangan**. Zoom menggunakan nearest-neighbor sehingga tidak membuat ketajaman interpolasi palsu. Sampel dipilih dengan seed tetap dan sama antar-arm.

CSV review sengaja kosong: isi label target, realisme tekstur, batas mask, keputusan dan reviewer. Edge energy hanya diagnostik; noise juga dapat menaikkannya. Jangan menerima kandidat secara otomatis dari nilai tersebut. Panel train ini belum membuktikan generalisasi atau peningkatan AP.


In [ ]:
from IPython.display import display, Image as DisplayImage
PANELDIR = WORK/'texture_panels'
for job in JOBS:
    for epoch in PANEL_EPOCHS:
        for weights in PANEL_WEIGHTS:
            target=PANELDIR/f'{job["name"]}_{epoch}_{weights}'
            run(HERE/'tools/texture_panel.py','--crops',CROPS,'--checkpoints-dir',CKPT,
                '--name',job['name'],'--epoch',epoch,'--weights',weights,'--out',target,
                '--device','cuda','--count',12,'--seed',2026,'--patch-size',TEXTURE_SIZE)
            print(target.name)
            display(DisplayImage(filename=str(target/'panel_00.png')))
print('Semua halaman PNG dan review.csv tersimpan di',PANELDIR)


## Kurva loss dan ekspor kandidat full-frame opsional
Loss G_A/G_B sudah **termasuk** texture_G_A/G_B pada arm ROI; jangan menjumlahkannya lagi. Loss antararsitektur bukan peringkat kualitas. Tidak ada pemilihan checkpoint otomatis berdasarkan loss GAN.

Ekspor di bawah memakai checkpoint **latest** melalui exporter yang sudah ada, mencakup ripe train saja, dan tetap menghasilkan `needs_review`. Untuk checkpoint lain, lakukan penambahan dukungan epoch secara eksplisit sebelum ekspor; jangan mengganti nama checkpoint lama. Kandidat baru perlu review sebelum notebook balancing/detektor berikutnya. Test lama tidak dijalankan kembali oleh notebook ini.


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
for job in JOBS:
    history=json.loads((CKPT/job['name']/'loss.json').read_text())['history']
    if not history: continue
    axes[0].plot([r['iters'] for r in history],[r['G_B'] for r in history],label=job['name'])
    if 'texture_G_B' in history[0]:
        axes[1].plot([r['iters'] for r in history],[r['texture_G_B'] for r in history],label=job['name'])
for ax,title in zip(axes,['G_B adversarial (termasuk local jika aktif)','Local G_B weighted loss']):
    ax.set_title(title);ax.set_xlabel('training images');ax.legend()
fig.tight_layout();fig.savefig(WORK/'training_losses.png',dpi=150)
plt.show()

EXPORT_FULLFRAME = False
EXPORT_WEIGHTS = 'raw'
if EXPORT_FULLFRAME:
    for job in JOBS:
        target=WORK/'fullframe_candidates'/f'{job["name"]}_{EXPORT_WEIGHTS}'
        run(HERE/'tools/generate_balance_candidates.py','--checkpoints-dir',CKPT,
            '--name',job['name'],'--weights',EXPORT_WEIGHTS,'--crops',CROPS,
            '--dataset',DATASET,'--device','cuda','--out',target)
        run(HERE/'tools/review.py','export','--bundle',target)
        run(HERE/'tools/review.py','tool','--bundle',target)
print('Output eksperimen:',WORK)


## Interpretasi dan kelanjutan

- **Prioritas 1:** bandingkan baseline vs ROI pada input sama: tekstur tampak wajar seperti unripe nyata, tidak berupa pola berulang/noise, label tetap unripe, bentuk dan batas mask tidak rusak.
- **Prioritas 2:** ulangi full dengan empat arm. ROI+SK vs ROI mengukur tambahan SK; SK vs baseline memisahkan kontribusinya. Semua mulai dari awal dengan seed dan jadwal sama.
- Pertahankan jumlah sintetik dan prosedur review yang sebanding saat menilai detektor. Jangan mengubah arsitektur YOLO bersamaan dengan perubahan GAN.
- Ukur AP50:95 unripe, mAP50:95, AP ripe dan recall pada validasi nyata; replikasi generator dan detektor untuk kandidat final. Keberhasilan visual tidak menjamin kenaikan AP.
- Keterbatasan resolusi: discriminator lokal tidak dapat mengembalikan detail yang sudah hilang ketika foto asli dikecilkan ke crop 256. Eksperimen 512 harus menjadi studi terpisah dengan crop pool baru, bukan mengganti digest pool ini.
- Notebook tidak melakukan auto-resume: trainer lama belum menyimpan optimizer, scaler dan RNG untuk resume identik. Run terputus harus diberi nama baru dan dicatat sebagai run baru.

### Landasan desain
- [SCU-CGAN, arXiv 2025](https://arxiv.org/abs/2512.08362): discriminator target region dan preservasi latar; penerapan di sini memakai patch interior grayscale dan ResNet, bukan reproduksi SCU-CGAN.
- [Tea CycleGAN, Agriculture 2025](https://doi.org/10.3390/agriculture15161739): fitur multi-skala dan preservasi penempatan objek; SK-inspired block di sini lebih kecil.
- [TC-CycleGAN, Sensors 2026](https://doi.org/10.3390/s26020395): augmentasi kelas langka dan detail objek; arsitektur TC-UNet/ContraNorm tidak diimplementasikan dalam eksperimen ini.
